# Experimento principal

## Imports necesarios

In [5]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, hinge_loss
from joblib import Parallel, delayed

## Funciones 

In [6]:


def procesar_dataset(archivo, epocas):
    """
    Función objetivo para joblib. Procesa un único dataset completo.
    Retorna el nombre del archivo, las métricas, las estadísticas y posibles errores.
    """
    if not os.path.exists(archivo):
        return archivo, None, None, f"[ERROR] Archivo no encontrado: {archivo}"

    nombre_esquema = archivo.replace('.csv', '')
    os.makedirs(nombre_esquema, exist_ok=True)

    df = pd.read_csv(archivo)
    X = df.drop(columns=['clase']).values
    y = df['clase'].values
    clases_unicas = np.unique(y)

    metricas = {'acc': [], 'recall': [], 'f1': [], 'auc': []}
    sss_test = StratifiedShuffleSplit(n_splits=10, test_size=0.15, random_state=42)

    fold = 1
    for train_val_idx, test_idx in sss_test.split(X, y):
        X_train_val, X_test = X[train_val_idx], X[test_idx]
        y_train_val, y_test = y[train_val_idx], y[test_idx]

        sss_val = StratifiedShuffleSplit(n_splits=1, test_size=0.17647, random_state=42)
        for train_idx, val_idx in sss_val.split(X_train_val, y_train_val):
            X_train, X_val = X_train_val[train_idx], X_train_val[val_idx]
            y_train, y_val = y_train_val[train_idx], y_train_val[val_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)

        svm = SGDClassifier(loss='hinge', random_state=42, warm_start=True)
        train_losses, val_losses = [], []

        for epoch in range(epocas):
            svm.partial_fit(X_train, y_train, classes=clases_unicas)

            df_train = svm.decision_function(X_train)
            df_val = svm.decision_function(X_val)

            loss_t = hinge_loss(y_train, df_train)
            loss_v = hinge_loss(y_val, df_val)

            train_losses.append(loss_t)
            val_losses.append(loss_v)

        plt.figure(figsize=(8, 5))
        plt.plot(range(1, epocas+1), train_losses, label='Train Loss', color='blue')
        plt.plot(range(1, epocas+1), val_losses, label='Val Loss', color='orange')
        plt.title(f'Loss vs Epoch - Fold {fold}\n{nombre_esquema}')
        plt.xlabel('Epoch')
        plt.ylabel('Hinge Loss')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        ruta_grafica = os.path.join(nombre_esquema, f'loss_fold_{fold}.png')
        plt.savefig(ruta_grafica)
        plt.close()

        y_pred = svm.predict(X_test)
        y_scores = svm.decision_function(X_test)

        metricas['acc'].append(accuracy_score(y_test, y_pred))
        metricas['recall'].append(recall_score(y_test, y_pred))
        metricas['f1'].append(f1_score(y_test, y_pred))
        metricas['auc'].append(roc_auc_score(y_test, y_scores))

        fold += 1

    df_metricas = pd.DataFrame(metricas)
    df_metricas.index = [f"Fold {i+1}" for i in range(10)]

    stats = pd.DataFrame({
        'Media': df_metricas.mean(),
        'Mediana': df_metricas.median(),
        'Desv. Est.': df_metricas.std()
    }).T

    return archivo, df_metricas, stats, None

def entrenar_evaluar_svm_paralelo():
    archivos = [
        "radiomics_DE_rand_1.csv",
        "radiomics_DE_best_1.csv",
        "radiomics_DE_rand_2.csv",
        "radiomics_DE_best_2.csv"
    ]
    epocas = 50

    print(f"[INFO] Iniciando procesamiento en paralelo para {len(archivos)} datasets con joblib...")
    print(f"[INFO] Utilizando todos los núcleos disponibles (n_jobs=-1).")

    # Ejecución concurrente. Cada archivo se procesa en paralelo.
    resultados = Parallel(n_jobs=-1)(delayed(procesar_dataset)(archivo, epocas) for archivo in archivos)

    # Impresión secuencial de los resultados recolectados
    for archivo, df_metricas, stats, error in resultados:
        if error:
            print(error)
            continue

        print(f"\n{'-'*60}")
        print(f"[INFO] Resultados Dataset: {archivo}")
        print(f"{'-'*60}")
        print(f"\n[RESULTADOS] 10 Folds:")
        print(df_metricas.round(4).to_string())

        print(f"\n[ESTADISTICAS] Finales ({archivo}):")
        print(stats.round(4).to_string())
        print("\n")


In [7]:
if __name__ == "__main__":
    entrenar_evaluar_svm_paralelo()

[INFO] Iniciando procesamiento en paralelo para 4 datasets con joblib...
[INFO] Utilizando todos los núcleos disponibles (n_jobs=-1).

------------------------------------------------------------
[INFO] Resultados Dataset: radiomics_DE_rand_1.csv
------------------------------------------------------------

[RESULTADOS] 10 Folds:
            acc  recall      f1     auc
Fold 1   0.9352  0.9704  0.9562  0.9581
Fold 2   0.9147  0.9610  0.9426  0.9614
Fold 3   0.9204  0.9438  0.9453  0.9591
Fold 4   0.9158  0.9423  0.9423  0.9514
Fold 5   0.9124  0.9376  0.9398  0.9453
Fold 6   0.9215  0.9828  0.9481  0.9658
Fold 7   0.9363  0.9548  0.9562  0.9732
Fold 8   0.8885  0.9610  0.9263  0.9220
Fold 9   0.9317  0.9626  0.9536  0.9687
Fold 10  0.9215  0.9766  0.9478  0.9684

[ESTADISTICAS] Finales (radiomics_DE_rand_1.csv):
               acc  recall      f1     auc
Media       0.9198  0.9593  0.9458  0.9573
Mediana     0.9209  0.9610  0.9465  0.9603
Desv. Est.  0.0139  0.0150  0.0090  0.0150



--